# Evaluation
Évaluation détaillée du modèle retenu (Random Forest) sur le dernier split temporel.

## Imports et chargement

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

os.makedirs("../reports_ml", exist_ok=True)

df = pd.read_csv("../data/processed/dataset_final.csv")
df["DateHeure"] = pd.to_datetime(df["DateHeure"])
df = df.sort_values("DateHeure").reset_index(drop=True)

FEATURES = [
    "Temperature",
    "Humidite",
    "Pression",
    "VitesseVent",
    "Heure_sin",
    "Heure_cos",
    "Mois_sin",
    "Mois_cos",
    "IsWeekend",
    "wind_dir_sin",
    "wind_dir_cos",
    "cloud_cover",
    "precipitation_bin",
    "AQI_mean_6h",
] + [c for c in df.columns if c.startswith("ville_")]

TARGET = "AqiGlobal"
X = df[FEATURES]
y = df[TARGET]

best_model = joblib.load("../src/ml/models/aqi_model.pkl")
tscv = TimeSeriesSplit(n_splits=3)

print(f"Modèle chargé : {type(best_model).__name__}")
print(f"Dataset       : {len(df)} lignes. {len(FEATURES)} features")

Modèle chargé : RandomForestRegressor
Dataset       : 10526 lignes. 25 features


## 1. Métriques sur le dernier split test

In [2]:
# Dernier split = données les plus récentes = le plus réaliste
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}  (erreur moyenne en points AQI)")
print(f"RMSE : {rmse:.2f} (pénalise les grosses erreurs)")
print(f"R²   : {r2:.4f}  (% de variance expliquée)")
print()
print("Seuils d acceptation :")
print(
    f"  R²   > 0.5 (jury) : {'Bon' if r2 > 0.5 else 'Mauvais'}  |  Objectif > 0.8 : {'Atteint' if r2 > 0.8 else 'Non atteint'}"
)
print(
    f"  MAE  < 10         : {'Bon' if mae < 10 else 'Mauvais'}  |  Objectif < 5  : {'Atteint' if mae < 5 else 'Non atteint'}"
)
print(
    f"  RMSE < 15         : {'Bon' if rmse < 15 else 'Mauvais'}  |  Objectif < 10 : {'Atteint' if rmse < 10 else 'Non atteint'}"
)

MAE  : 4.25  (erreur moyenne en points AQI)
RMSE : 6.42 (pénalise les grosses erreurs)
R²   : 0.8711  (% de variance expliquée)

Seuils d acceptation :
  R²   > 0.5 (jury) : Bon  |  Objectif > 0.8 : Bon
  MAE  < 10         : Bon  |  Objectif < 5  : Bon
  RMSE < 15         : Bon  |  Objectif < 10 : Bon


### Observations - Métriques
Tous les seuils sont atteints. y compris les objectifs ambitieux.

**R² de 0.8711** le modèle explique 87% de la variance de l'AQI sur la période de test juin-juillet. C'est le meilleur score des trois splits ce qui confirme que le modèle s'améliore avec plus de données d'entraînement. Largement au-dessus de l'exigence jury de 0.5.

**MAE de 4.25 points AQI** en moyenne le modèle se trompe de 4.25 points sur l'échelle AQI. Pour donner un ordre de grandeur : sur un AQI réel de 40 (qualité bonne). le modèle prédit entre 36 et 44. C'est une précision très satisfaisante pour un usage d'alerte.

**RMSE de 6.42 points AQI** légèrement supérieur à la MAE. ce qui indique que les grosses erreurs sont rares mais existent. L'écart entre MAE (4.25) et RMSE (6.42) est modéré. le modèle n'a pas de très grands pics d'erreur.
Conclusion : le modèle dépasse tous les objectifs fixés. C'est un excellent résultat pour un MVP entraîné sur 3 mois de données avec des trous temporels.

## 2. Précision des alertes (métrique métier)

In [3]:
SEUIL_ALERTE = 100

alertes_reelles = y_test > SEUIL_ALERTE
alertes_predites = y_pred > SEUIL_ALERTE

precision_alertes = (alertes_reelles == alertes_predites).mean() * 100
faux_negatifs = (alertes_reelles & ~alertes_predites).sum()
faux_positifs = (~alertes_reelles & alertes_predites).sum()
vrais_positifs = (alertes_reelles & alertes_predites).sum()

print(f"Précision des alertes          : {precision_alertes:.1f}%")
print(f"Alertes correctement détectées : {vrais_positifs}")
print(f"Alertes manquées (faux négatifs) : {faux_negatifs}  ← les plus graves")
print(f"Fausses alertes  (faux positifs) : {faux_positifs}")
print()
print("Note : dans un contexte santé/environnement.")
print("un faux négatif (alerte manquée) est plus grave qu un faux positif.")

Précision des alertes          : 100.0%
Alertes correctement détectées : 0
Alertes manquées (faux négatifs) : 0  ← les plus graves
Fausses alertes  (faux positifs) : 0

Note : dans un contexte santé/environnement.
un faux négatif (alerte manquée) est plus grave qu un faux positif.


### Observations - Précision des alertes
> *(à compléter)*

## 3. Feature Importance

In [ ]:
importance = pd.Series(
    best_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=True).tail(10)

plt.figure(figsize=(10, 6))
importance.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 features les plus importantes')
plt.xlabel('Importance relative')
plt.tight_layout()
plt.savefig('../reports_ml/feature_importance.png')
plt.show()

print('Top 10 features :')
print(importance[::-1].round(4))

### Observations - Feature Importance
> *(à compléter)*

## 4. Courbe Prévision vs Réel

In [ ]:
results = pd.DataFrame({
    'DateHeure' : df.iloc[test_idx]['DateHeure'].values,
    'AQI_Réel'  : y_test.values,
    'AQI_Prédit': y_pred
})

plt.figure(figsize=(14, 5))
plt.plot(results['DateHeure'], results['AQI_Réel'],
         label='AQI Réel', color='steelblue', alpha=0.7)
plt.plot(results['DateHeure'], results['AQI_Prédit'],
         label='AQI Prédit', color='orange', linestyle='--')
plt.axhline(y=100, color='red', linestyle=':', label='Seuil alerte (100)')
plt.title('AQI Réel vs Prédit — Période de test (juin-juillet 2026)')
plt.xlabel('Date')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.savefig('../reports_ml/prevision_vs_reel.png')
plt.show()

### Observations — Courbe Prévision vs Réel
> *(à compléter)*

## 5. Résumé d évaluation

In [ ]:
print('=' * 45)
print('RÉSUMÉ D ÉVALUATION — GoodAir AQI Model')
print('=' * 45)
print(f'Modèle             : {type(best_model).__name__}')
print(f'R²                 : {r2:.4f}')
print(f'MAE                : {mae:.2f} points AQI')
print(f'RMSE               : {rmse:.2f} points AQI')
print(f'Précision alertes  : {precision_alertes:.1f}%')
print(f'Alertes manquées   : {faux_negatifs}')
print(f'Fausses alertes    : {faux_positifs}')
print(f'Dataset            : {len(df)} lignes. {len(FEATURES)} features')
print(f'Période            : mars 2026 → juillet 2026')
print(f'Biais assumé       : saisonnalité printemps/été uniquement')
print(f'Continuous Training: ré-entraînement prévu tous les 3 mois')

### Observations — Résumé
> *(à compléter)*